# Tabella 2 — Linear evaluation with frozen features (Sezione 3.2)

Riproduce **Tabella 2a** (linear probe Top-1 su ImageNet, with vs without registers) per i tre modelli e **Tabella 2b** (zero-shot classification, solo OpenCLIP).

Il backbone resta frozen; sopra si fitta solo un classificatore lineare (`sklearn.LogisticRegression`) sui CLS features.

Subset stratificato: 50 classi × 10 img da ImageNet val (HF), split 70/30 train/val.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT.name != "Poli" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("package root:", ROOT)

## 1. Scarica il subset ImageNet (skip se già fatto)

In [ ]:
from data_loaders.imagenet import download_subset, ImageNetSubset
meta = download_subset(num_classes=50, images_per_class=10)
meta

## 2. Tabella 2a — Linear probe ImageNet

In [ ]:
import torch
from models import load_dinov2, load_openclip, load_deit3
from eval.linear_probe import run_linear_probe, make_transform

DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print("device:", DEVICE)

In [ ]:
configs = [
    ("DINOv2 ViT-L/14",        lambda: load_dinov2(with_registers=False, img_size=518), 518),
    ("DINOv2 ViT-L/14 +reg4",  lambda: load_dinov2(with_registers=True,  img_size=518), 518),
    ("OpenCLIP ViT-B/16",       lambda: load_openclip(with_registers=False), 224),
    ("OpenCLIP +tt-reg4",       lambda: load_openclip(with_registers=True, num_registers=4), 224),
    ("DeiT-III ViT-B/16",       lambda: load_deit3(with_registers=False), 224),
    ("DeiT-III +reg4 (inj.)",   lambda: load_deit3(with_registers=True,  num_registers=4), 224),
]

results = {}
for label, factory, img_size in configs:
    print(f"\n--- {label} ---")
    model = factory()
    tfm = make_transform(img_size)
    train_ds = ImageNetSubset("train", transform=tfm)
    val_ds   = ImageNetSubset("val",   transform=tfm)
    res = run_linear_probe(model, train_ds, val_ds, device=DEVICE, batch_size=4)
    print(f"  top1 = {res['top1']:.3f}  (n_train={res['n_train']}, n_val={res['n_val']}, dim={res['feature_dim']})")
    results[label] = res
    del model

In [ ]:
import pandas as pd
df = pd.DataFrame(results).T[["top1", "feature_dim", "n_train", "n_val"]]
df["top1"] = (df["top1"] * 100).round(2)
df

## 3. Tabella 2b — Zero-shot OpenCLIP (TODO)

Implementeremo nel prossimo step. Richiede l'encoder testuale di OpenCLIP + i template di prompt ImageNet.